# Práctica de clase 5: Pipelines de Whisper y Huggin Face
Objetivos de la práctica:
1. Utilizar el pipeline de Whisper para transcribir audio en español.
2. Probar varios modelos de Hugging Face para tareas diferentes:
   - ASR / reconocimiento automático de voz
   - traducción
   - resumen
   - generación de texto
3. Construir una aplicación simple con pipelines.
4. Guardar resultados en archivos CSV y Markdown para análisis posterior.

Para audio MP3 puede ser necesario instalar ffmpeg.
En Windows puedes instalarlo desde:
    https://www.gyan.dev/ffmpeg/builds/

Estructura sugerida de carpetas:
    practica_clase_5_whisper_huggingface_pipelines.py
    audios/
        ejemplo_1.wav
        ejemplo_2.mp3
    referencias.csv   (opcional)

In [1]:
# ============================================================
# 1. IMPORTACIÓN DE LIBRERÍAS
# ============================================================

from __future__ import annotations

import csv
import json
import os
import sys
import time
import traceback
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Dict, List, Optional, Tuple, Any

import numpy as np
import pandas as pd
import torch
from transformers import pipeline

c:\Users\jpach\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# ============================================================
# 2. CONFIGURACIÓN GENERAL
# ============================================================

BASE_DIR = Path.cwd()
AUDIO_DIR = BASE_DIR / "audios"
OUTPUT_DIR = BASE_DIR / "salidas_practica_clase5"
OUTPUT_DIR.mkdir(exist_ok=True)
AUDIO_DIR.mkdir(exist_ok=True)

# Modelo principal de ASR. Para clase, tiny es más rápido.
WHISPER_MODEL = "openai/whisper-tiny"

# Modelos de traducción. Si alguno falla, el script intenta el siguiente.
TRANSLATION_ES_EN_CANDIDATES = [
    "Helsinki-NLP/opus-mt-es-en",
]

TRANSLATION_EN_ES_CANDIDATES = [
    "Helsinki-NLP/opus-mt-en-es",
]

# Modelos de resumen. Algunos modelos de resumen en español pueden cambiar o ser pesados.
# El script intenta modelos y si no puede cargarlos usa un resumen extractivo local.
SUMMARIZATION_CANDIDATES = [
    "mrm8488/bert2bert_shared-spanish-finetuned-summarization",
    "IIC/mt5-small-spanish-summarization",
    "facebook/bart-large-cnn",
]

# Modelos de generación de texto. Para español se intenta primero un GPT-2 español.
TEXT_GENERATION_CANDIDATES = [
    "datificate/gpt2-small-spanish",
    "DeepESP/gpt2-spanish",
    "distilgpt2",
]

# Parámetros de ejecución
MAX_AUDIO_FILES = 5
MAX_TEXT_LENGTH = 800
MAX_SUMMARY_LENGTH = 120
MIN_SUMMARY_LENGTH = 25
MAX_GENERATED_TOKENS = 80

# Activar o desactivar secciones.
RUN_ASR = True
RUN_TRANSLATION = True
RUN_SUMMARIZATION = True
RUN_TEXT_GENERATION = True
RUN_SIMPLE_APP = True  # Cambia a True para activar la app interactiva al final.

# En CPU puede ser útil limitar hilos.
torch.set_num_threads(1)

In [3]:
# ============================================================
# 3. UTILIDADES GENERALES
# ============================================================

@dataclass
class PipelineResult:
    task: str
    model: str
    input_data: str
    output_data: str
    runtime_seconds: float
    success: bool
    error: str = ""


def get_device() -> int:
    """
    Devuelve 0 si hay GPU CUDA disponible, -1 para CPU.
    La función pipeline de Hugging Face usa este formato.
    """
    return 0 if torch.cuda.is_available() else -1


def print_header(title: str) -> None:
    print("\n" + "=" * 78)
    print(title)
    print("=" * 78)


def safe_text(text: str, max_chars: int = 1600) -> str:
    text = str(text).replace("\n", " ").strip()
    if len(text) > max_chars:
        return text[:max_chars] + "..."
    return text


def save_json(path: Path, data: Any) -> None:
    with path.open("w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)


def save_results_csv(path: Path, results: List[PipelineResult]) -> None:
    rows = [asdict(r) for r in results]
    df = pd.DataFrame(rows)
    df.to_csv(path, index=False, encoding="utf-8-sig")


def list_audio_files(audio_dir: Path) -> List[Path]:
    extensions = ["*.wav", "*.mp3", "*.m4a", "*.flac", "*.ogg"]
    files: List[Path] = []
    for ext in extensions:
        files.extend(audio_dir.glob(ext))
    return sorted(files)[:MAX_AUDIO_FILES]


def load_optional_references(path: Path) -> Dict[str, str]:
    """
    Carga referencias opcionales para calcular WER.
    Formato: archivo,texto_referencia
    """
    if not path.exists():
        return {}

    refs: Dict[str, str] = {}
    with path.open("r", encoding="utf-8-sig", newline="") as f:
        reader = csv.DictReader(f)
        for row in reader:
            filename = row.get("archivo", "").strip()
            text = row.get("texto_referencia", "").strip()
            if filename and text:
                refs[filename] = text
    return refs

In [4]:
# ============================================================
# 4. MÉTRICA WER DESDE CERO
# ============================================================

def normalize_for_wer(text: str) -> List[str]:
    import re
    text = text.lower().strip()
    text = re.sub(r"[^a-záéíóúüñ0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.split()


def word_error_rate(reference: str, hypothesis: str) -> float:
    """
    Calcula Word Error Rate con distancia de Levenshtein a nivel palabra.
    WER = (S + D + I) / N
    """
    ref = normalize_for_wer(reference)
    hyp = normalize_for_wer(hypothesis)

    if len(ref) == 0:
        return 0.0 if len(hyp) == 0 else 1.0

    dp = np.zeros((len(ref) + 1, len(hyp) + 1), dtype=int)

    for i in range(len(ref) + 1):
        dp[i, 0] = i
    for j in range(len(hyp) + 1):
        dp[0, j] = j

    for i in range(1, len(ref) + 1):
        for j in range(1, len(hyp) + 1):
            if ref[i - 1] == hyp[j - 1]:
                cost = 0
            else:
                cost = 1
            dp[i, j] = min(
                dp[i - 1, j] + 1,       # borrado
                dp[i, j - 1] + 1,       # inserción
                dp[i - 1, j - 1] + cost # sustitución
            )

    return dp[len(ref), len(hyp)] / len(ref)

In [5]:
# ============================================================
# 5. CARGA SEGURA DE PIPELINES
# ============================================================

def load_pipeline_with_candidates(
    task: str,
    candidates: List[str],
    device: int,
    extra_kwargs: Optional[Dict[str, Any]] = None,
):
    """
    Intenta cargar un pipeline con varios modelos candidatos.
    Devuelve (pipeline, model_name) o (None, "") si todos fallan.
    """
    extra_kwargs = extra_kwargs or {}

    for model_name in candidates:
        try:
            print(f"Cargando pipeline '{task}' con modelo: {model_name}")
            start = time.perf_counter()

            pipe = pipeline(
                task,
                model=model_name,
                device=device,
                **extra_kwargs,
            )

            elapsed = time.perf_counter() - start
            print(f"Pipeline cargado correctamente en {elapsed:.2f} s")
            return pipe, model_name

        except Exception as e:
            print(f"No se pudo cargar {model_name}")
            print("Motivo:", str(e)[:400])

    print(f"No se pudo cargar ningún modelo para la tarea: {task}")
    return None, ""

In [6]:
# ============================================================
# 6. SECCIÓN A: ASR CON WHISPER
# ============================================================

def run_whisper_asr(audio_files: List[Path], references: Dict[str, str]) -> List[PipelineResult]:
    print_header("SECCIÓN A - Transcripción de audio en español con Whisper")

    results: List[PipelineResult] = []

    if not audio_files:
        print(
            "No se encontraron archivos de audio en la carpeta 'audios/'.\n"
            "Agrega archivos .wav, .mp3, .m4a, .flac u .ogg y vuelve a ejecutar.\n"
            "La práctica continuará con las secciones de texto."
        )
        return results

    device = get_device()

    # torch_dtype ayuda en GPU; en CPU usamos float32 para evitar incompatibilidades.
    model_kwargs = {}
    if torch.cuda.is_available():
        model_kwargs["torch_dtype"] = torch.float16

    asr_pipe, loaded_model = load_pipeline_with_candidates(
        task="automatic-speech-recognition",
        candidates=[WHISPER_MODEL],
        device=device,
        extra_kwargs={"model_kwargs": model_kwargs} if model_kwargs else {},
    )

    if asr_pipe is None:
        return results

    for audio_path in audio_files:
        print(f"\nTranscribiendo: {audio_path.name}")
        start = time.perf_counter()

        try:
            # Primero se intenta configurar idioma y tarea.
            try:
                output = asr_pipe(
                    str(audio_path),
                    generate_kwargs={"language": "spanish", "task": "transcribe"},
                    return_timestamps=False,
                )
            except TypeError:
                # Compatibilidad con versiones antiguas.
                output = asr_pipe(str(audio_path))

            elapsed = time.perf_counter() - start
            text = output["text"] if isinstance(output, dict) and "text" in output else str(output)

            print("Transcripción:")
            print(text)

            # Calcular WER si hay referencia.
            ref = references.get(audio_path.name)
            if ref:
                wer = word_error_rate(ref, text)
                print(f"WER contra referencia: {wer:.4f}")
                text_to_save = f"{text}\n\n[Referencia: {ref}]\n[WER: {wer:.4f}]"
            else:
                text_to_save = text

            results.append(
                PipelineResult(
                    task="ASR Whisper",
                    model=loaded_model,
                    input_data=audio_path.name,
                    output_data=text_to_save,
                    runtime_seconds=elapsed,
                    success=True,
                )
            )

        except Exception as e:
            elapsed = time.perf_counter() - start
            print("Error durante ASR:", e)
            results.append(
                PipelineResult(
                    task="ASR Whisper",
                    model=loaded_model,
                    input_data=audio_path.name,
                    output_data="",
                    runtime_seconds=elapsed,
                    success=False,
                    error=str(e),
                )
            )

    return results

In [7]:
# ============================================================
# 7. SECCIÓN B: TRADUCCIÓN CON PIPELINES
# ============================================================

def run_translation_examples(texts_es: List[str]) -> List[PipelineResult]:
    print_header("SECCIÓN B - Traducción español → inglés e inglés → español")

    results: List[PipelineResult] = []
    device = get_device()

    if not texts_es:
        texts_es = [
            "Whisper es un modelo de reconocimiento de voz basado en Transformers.",
            "La biblioteca Hugging Face permite cargar modelos preentrenados con pocas líneas de código.",
            "Los pipelines simplifican la inferencia para tareas como traducción y resumen.",
        ]

    es_en_pipe, es_en_model = load_pipeline_with_candidates(
        task="translation",
        candidates=TRANSLATION_ES_EN_CANDIDATES,
        device=device,
    )

    en_es_pipe, en_es_model = load_pipeline_with_candidates(
        task="translation",
        candidates=TRANSLATION_EN_ES_CANDIDATES,
        device=device,
    )

    for text in texts_es[:3]:
        if es_en_pipe is not None:
            start = time.perf_counter()
            try:
                out = es_en_pipe(text, max_length=256)
                elapsed = time.perf_counter() - start
                translated = out[0].get("translation_text", str(out))
                print(f"\nES: {text}")
                print(f"EN: {translated}")
                results.append(PipelineResult("Traducción ES-EN", es_en_model, text, translated, elapsed, True))
            except Exception as e:
                elapsed = time.perf_counter() - start
                results.append(PipelineResult("Traducción ES-EN", es_en_model, text, "", elapsed, False, str(e)))

        # Ejemplo inverso fijo para mostrar segunda dirección.
        english_text = "Transformers are useful for audio, text and vision tasks."
        if en_es_pipe is not None:
            start = time.perf_counter()
            try:
                out = en_es_pipe(english_text, max_length=256)
                elapsed = time.perf_counter() - start
                translated = out[0].get("translation_text", str(out))
                print(f"\nEN: {english_text}")
                print(f"ES: {translated}")
                results.append(PipelineResult("Traducción EN-ES", en_es_model, english_text, translated, elapsed, True))
                break
            except Exception as e:
                elapsed = time.perf_counter() - start
                results.append(PipelineResult("Traducción EN-ES", en_es_model, english_text, "", elapsed, False, str(e)))
                break

    return results

In [8]:
# ============================================================
# 8. SECCIÓN C: RESUMEN
# ============================================================

def extractive_summary_local(text: str, max_sentences: int = 2) -> str:
    """
    Resumen extractivo simple de respaldo.
    Selecciona las oraciones más informativas según frecuencia de palabras.
    No sustituye a un modelo neuronal, pero permite continuar la práctica si
    no se logra descargar un modelo de resumen.
    """
    import re
    sentences = re.split(r"(?<=[.!?])\s+", text.strip())
    sentences = [s.strip() for s in sentences if len(s.strip()) > 0]

    if len(sentences) <= max_sentences:
        return text.strip()

    words = re.findall(r"\b\w+\b", text.lower(), flags=re.UNICODE)
    stopwords = {
        "el", "la", "los", "las", "un", "una", "unos", "unas", "de", "del",
        "y", "o", "a", "en", "con", "para", "por", "que", "se", "es", "son",
        "como", "su", "sus", "al", "lo", "más", "menos", "este", "esta", "estos",
        "estas", "también", "puede", "pueden", "entre", "sobre", "desde"
    }
    freq: Dict[str, int] = {}
    for w in words:
        if w not in stopwords and len(w) > 2:
            freq[w] = freq.get(w, 0) + 1

    scores = []
    for idx, sentence in enumerate(sentences):
        s_words = re.findall(r"\b\w+\b", sentence.lower(), flags=re.UNICODE)
        score = sum(freq.get(w, 0) for w in s_words)
        scores.append((score, idx, sentence))

    best = sorted(scores, reverse=True)[:max_sentences]
    best_ordered = sorted(best, key=lambda x: x[1])
    return " ".join(sentence for _, _, sentence in best_ordered)


def run_summarization_examples(texts: List[str]) -> List[PipelineResult]:
    print_header("SECCIÓN C - Resumen con modelos de Hugging Face o respaldo local")

    results: List[PipelineResult] = []
    device = get_device()

    long_text = (
        "Whisper es un modelo de reconocimiento automático de voz basado en una arquitectura Transformer "
        "encoder-decoder. Su diseño permite transcribir audio, identificar idioma y realizar traducción de voz "
        "mediante el uso de tokens especiales que controlan la tarea. Por otra parte, Hugging Face proporciona "
        "un ecosistema de herramientas como transformers, datasets, tokenizers, accelerate y pipelines, que facilitan "
        "la carga de modelos preentrenados, el procesamiento de datos y la inferencia rápida. En conjunto, estas "
        "herramientas permiten que estudiantes e investigadores pasen de la teoría a prototipos funcionales de manera "
        "más rápida y reproducible."
    )

    candidate_text = " ".join(texts) if texts else long_text
    candidate_text = safe_text(candidate_text, MAX_TEXT_LENGTH)

    summarizer, model_name = load_pipeline_with_candidates(
        task="summarization",
        candidates=SUMMARIZATION_CANDIDATES,
        device=device,
    )

    if summarizer is not None:
        start = time.perf_counter()
        try:
            out = summarizer(
                candidate_text,
                max_length=MAX_SUMMARY_LENGTH,
                min_length=MIN_SUMMARY_LENGTH,
                do_sample=False,
            )
            elapsed = time.perf_counter() - start
            summary = out[0].get("summary_text", str(out))
            print("\nResumen generado por modelo:")
            print(summary)
            results.append(PipelineResult("Resumen", model_name, candidate_text, summary, elapsed, True))
            return results
        except Exception as e:
            elapsed = time.perf_counter() - start
            print("Falló el modelo de resumen. Se usará respaldo local.")
            results.append(PipelineResult("Resumen", model_name, candidate_text, "", elapsed, False, str(e)))

    start = time.perf_counter()
    summary = extractive_summary_local(candidate_text, max_sentences=2)
    elapsed = time.perf_counter() - start
    print("\nResumen extractivo local:")
    print(summary)
    results.append(PipelineResult("Resumen extractivo local", "local", candidate_text, summary, elapsed, True))
    return results

In [9]:
# ============================================================
# 9. SECCIÓN D: GENERACIÓN DE TEXTO
# ============================================================

def run_text_generation_examples() -> List[PipelineResult]:
    print_header("SECCIÓN D - Generación de texto con pipeline")

    results: List[PipelineResult] = []
    device = get_device()

    generator, model_name = load_pipeline_with_candidates(
        task="text-generation",
        candidates=TEXT_GENERATION_CANDIDATES,
        device=device,
    )

    if generator is None:
        return results

    prompts = [
        "Los modelos Transformer son importantes porque",
        "Una aplicación educativa de Whisper consiste en",
        "Hugging Face facilita el trabajo con inteligencia artificial porque",
    ]

    for prompt in prompts:
        start = time.perf_counter()
        try:
            out = generator(
                prompt,
                max_new_tokens=MAX_GENERATED_TOKENS,
                do_sample=True,
                temperature=0.8,
                top_p=0.9,
                num_return_sequences=1,
                pad_token_id=getattr(generator.tokenizer, "eos_token_id", None),
            )
            elapsed = time.perf_counter() - start
            generated = out[0].get("generated_text", str(out))
            print("\nPrompt:")
            print(prompt)
            print("Generación:")
            print(generated)
            results.append(PipelineResult("Generación de texto", model_name, prompt, generated, elapsed, True))
        except Exception as e:
            elapsed = time.perf_counter() - start
            results.append(PipelineResult("Generación de texto", model_name, prompt, "", elapsed, False, str(e)))

    return results

In [10]:
# ============================================================
# 10. SECCIÓN E: APLICACIÓN SIMPLE CON PIPELINES
# ============================================================

def simple_pipeline_app() -> None:
    """
    Aplicación simple de consola.
    Para activarla, cambia RUN_SIMPLE_APP = True al inicio del archivo.
    """
    print_header("SECCIÓN E - Aplicación simple con pipelines")
    print("Esta app permite probar tareas básicas desde consola.")
    print("Opciones: 1) traducción ES-EN, 2) resumen local, 3) generación de texto, 4) salir")

    device = get_device()
    es_en_pipe, es_en_model = load_pipeline_with_candidates(
        task="translation",
        candidates=TRANSLATION_ES_EN_CANDIDATES,
        device=device,
    )
    generator, gen_model = load_pipeline_with_candidates(
        task="text-generation",
        candidates=TEXT_GENERATION_CANDIDATES,
        device=device,
    )

    while True:
        print("\n" + "-" * 60)
        option = input("Selecciona opción [1-4]: ").strip()

        if option == "4":
            print("Saliendo de la app.")
            break

        if option not in {"1", "2", "3"}:
            print("Opción no válida.")
            continue

        text = input("Escribe el texto de entrada: ").strip()
        if not text:
            print("No se recibió texto.")
            continue

        if option == "1":
            if es_en_pipe is None:
                print("No hay pipeline de traducción disponible.")
                continue
            out = es_en_pipe(text, max_length=256)
            print("Traducción:", out[0].get("translation_text", out))

        elif option == "2":
            print("Resumen local:")
            print(extractive_summary_local(text, max_sentences=2))

        elif option == "3":
            if generator is None:
                print("No hay pipeline de generación disponible.")
                continue
            out = generator(
                text,
                max_new_tokens=MAX_GENERATED_TOKENS,
                do_sample=True,
                temperature=0.8,
                top_p=0.9,
                num_return_sequences=1,
                pad_token_id=getattr(generator.tokenizer, "eos_token_id", None),
            )
            print("Generación:")
            print(out[0].get("generated_text", out))


In [11]:
# ============================================================
# 11. REPORTE FINAL
# ============================================================

def build_markdown_report(results: List[PipelineResult]) -> str:
    lines: List[str] = []
    lines.append("# Reporte de práctica Clase 5: Whisper y Hugging Face\n")
    lines.append("## Objetivos\n")
    lines.append("- Usar Whisper para transcripción de audio en español.\n")
    lines.append("- Probar pipelines de traducción, resumen y generación.\n")
    lines.append("- Analizar ventajas, límites y tiempos de inferencia.\n")

    lines.append("## Resultados\n")
    for idx, r in enumerate(results, start=1):
        status = "correcto" if r.success else "falló"
        lines.append(f"### {idx}. {r.task} - {status}\n")
        lines.append(f"- Modelo: `{r.model}`\n")
        lines.append(f"- Tiempo: {r.runtime_seconds:.2f} s\n")
        if r.input_data:
            lines.append(f"- Entrada: {safe_text(r.input_data, 500)}\n")
        if r.output_data:
            lines.append(f"- Salida: {safe_text(r.output_data, 1000)}\n")
        if r.error:
            lines.append(f"- Error: {r.error}\n")
        lines.append("\n")

    lines.append("## Preguntas para análisis\n")
    questions = [
        "¿Qué diferencias observaste entre transcripción, traducción, resumen y generación?",
        "¿Qué ventajas tiene usar pipelines frente a implementar todo manualmente?",
        "¿Qué limitaciones aparecieron al trabajar con modelos preentrenados?",
        "¿Cómo influye el tamaño del modelo Whisper en tiempo y calidad?",
        "¿Qué riesgos existen al usar modelos descargados sin revisar su model card?",
        "¿Qué aplicación simple podrías construir combinando ASR + resumen + generación?",
    ]
    for q in questions:
        lines.append(f"- {q}\n")

    return "".join(lines)

In [12]:
# ============================================================
# 12. FUNCIÓN PRINCIPAL
# ============================================================

def main() -> None:
    print_header("PRÁCTICA CLASE 5 - Whisper y Hugging Face")
    print("Carpeta de audios:", AUDIO_DIR)
    print("Carpeta de salidas:", OUTPUT_DIR)
    print("Dispositivo:", "CUDA/GPU" if torch.cuda.is_available() else "CPU")

    all_results: List[PipelineResult] = []

    audio_files = list_audio_files(AUDIO_DIR)
    references = load_optional_references(BASE_DIR / "referencias.csv")

    if RUN_ASR:
        asr_results = run_whisper_asr(audio_files, references)
        all_results.extend(asr_results)
    else:
        asr_results = []

    # Tomar textos transcritos correctos como insumo para traducción/resumen.
    transcribed_texts = [r.output_data.split("\n\n[Referencia:")[0] for r in asr_results if r.success and r.output_data]

    if RUN_TRANSLATION:
        all_results.extend(run_translation_examples(transcribed_texts))

    if RUN_SUMMARIZATION:
        all_results.extend(run_summarization_examples(transcribed_texts))

    if RUN_TEXT_GENERATION:
        all_results.extend(run_text_generation_examples())

    # Guardar resultados.
    csv_path = OUTPUT_DIR / "resultados_pipelines_clase5.csv"
    json_path = OUTPUT_DIR / "resultados_pipelines_clase5.json"
    md_path = OUTPUT_DIR / "reporte_practica_clase5.md"

    save_results_csv(csv_path, all_results)
    save_json(json_path, [asdict(r) for r in all_results])
    md_path.write_text(build_markdown_report(all_results), encoding="utf-8")

    print_header("ARCHIVOS GENERADOS")
    print("CSV:", csv_path)
    print("JSON:", json_path)
    print("Reporte Markdown:", md_path)

    if RUN_SIMPLE_APP:
        simple_pipeline_app()
    else:
        print("\nLa app interactiva está incluida en el archivo, pero no se ejecutó.")
        print("Para activarla, cambia RUN_SIMPLE_APP = True al inicio del script.")

    print_header("PRÁCTICA FINALIZADA")
    print("Revisa los archivos de salida y responde las preguntas del reporte.")

In [13]:
if __name__ == "__main__":
    try:
        main()
    except KeyboardInterrupt:
        print("\nEjecución interrumpida por el usuario.")
    except Exception as exc:
        print("\nOcurrió un error general:", exc)


PRÁCTICA CLASE 5 - Whisper y Hugging Face
Carpeta de audios: c:\Users\jpach\Documents\Diplomado IA\Módulo X - Transformers\Module X - Transformers\Day_5\audios
Carpeta de salidas: c:\Users\jpach\Documents\Diplomado IA\Módulo X - Transformers\Module X - Transformers\Day_5\salidas_practica_clase5
Dispositivo: CPU

SECCIÓN A - Transcripción de audio en español con Whisper
Cargando pipeline 'automatic-speech-recognition' con modelo: openai/whisper-tiny



Device set to use cpu


Pipeline cargado correctamente en 49.01 s

Transcribiendo: audio_2026-07-16_20-38-17_1.wav
Error durante ASR: ffmpeg was not found but is required to load audio files from filename

Transcribiendo: audio_2026-07-16_20-38-26_1.wav
Error durante ASR: ffmpeg was not found but is required to load audio files from filename

Transcribiendo: audio_2026-07-16_20-38-30_1.wav
Error durante ASR: ffmpeg was not found but is required to load audio files from filename

Transcribiendo: audio_2026-07-16_20-38-33_1.wav
Error durante ASR: ffmpeg was not found but is required to load audio files from filename

SECCIÓN B - Traducción español → inglés e inglés → español
Cargando pipeline 'translation' con modelo: Helsinki-NLP/opus-mt-es-en


c:\Users\jpach\AppData\Local\Programs\Python\Python312\Lib\site-packages\transformers\models\marian\tokenization_marian.py:175: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")
Device set to use cpu


Pipeline cargado correctamente en 21.92 s
Cargando pipeline 'translation' con modelo: Helsinki-NLP/opus-mt-en-es


...

Pipeline cargado correctamente en 157.73 s

Prompt:
Los modelos Transformer son importantes porque
Generación:
Los modelos Transformer son importantes porque son una combinación de los mismos componentes.

El motor principal del Transformer es un turbocompresor. Los motores son turbocompresores de 4 tiempos con turbocompresor de 4 tiempos, pero la potencia máxima es de 560 kW (770 hp). El motor tiene una transmisión de 5 velocidades con un sistema de "pullback" de cinco velocidades con una velocidad máxima de 30

Prompt:
Una aplicación educativa de Whisper consiste en
Generación:
Una aplicación educativa de Whisper consiste en enseñar a los estudiantes sobre un tema específico, de manera que éstos puedan aprender de manera autónoma, y también para enseñar las lecciones de las matemáticas y de las ciencias de la computación.

Un estudiante puede aprender sobre un tema específico de un área, y puede aprender de manera autónoma.


La aplicación educativa de Whisper consiste en enseñar a l

Device set to use cpu


Pipeline cargado correctamente en 7.73 s
Cargando pipeline 'text-generation' con modelo: datificate/gpt2-small-spanish


Device set to use cpu


Pipeline cargado correctamente en 1.13 s

------------------------------------------------------------
Saliendo de la app.

PRÁCTICA FINALIZADA
Revisa los archivos de salida y responde las preguntas del reporte.
